<a href="https://colab.research.google.com/github/venkat4246/zepto-ai-ml-capstone-project/blob/main/analytics/02_eda.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import seaborn as sns

# Load the Titanic dataset ONCE
df = sns.load_dataset("titanic")

# Basic profiling
print("Dataset Information:")
df.info()

print("\nDataset Description:")
display(df.describe(include="all"))

print("\nDataset Shape:")
print(df.shape)

# Missing-value percentages
missing_pct = (df.isnull().sum() / len(df) * 100)
missing_pct = missing_pct[missing_pct > 0].sort_values(ascending=False)

print("\nMissing Value Percentages:")
print(missing_pct)

# Save the raw loaded dataset as the offline fallback
df.to_csv("titanic.csv", index=False)

print("\nSaved offline fallback as: titanic.csv")

In [ ]:
# Task 2 — Missing-value handling

df_clean = df.copy()

# Under 5% missing → drop those rows
df_clean = df_clean.dropna(subset=["embarked", "embark_town"])

# 5%–30% missing → impute with median
df_clean["age"] = df_clean["age"].fillna(df_clean["age"].median())

# More than 30% missing → drop the column
df_clean = df_clean.drop(columns=["deck"])

# Report the cleaning decisions
print("Missing-value handling:")
print("embarked: 0.224467% missing → rows dropped (<5%)")
print("embark_town: 0.224467% missing → rows dropped (<5%)")
print("age: 19.865320% missing → median imputation (5%-30%)")
print("deck: 77.216611% missing → column dropped (>30%)")

print("\nMissing values after cleaning:")
print(df_clean.isnull().sum())

print("\nCleaned dataset shape:")
print(df_clean.shape)

In [ ]:
# Task 3 — Univariate Analysis

import matplotlib.pyplot as plt
import pandas as pd

# Age histogram
plt.figure(figsize=(8, 5))
plt.hist(df_clean["age"], bins=20, edgecolor="black")
plt.title("Age Distribution")
plt.xlabel("Age")
plt.ylabel("Frequency")
plt.show()

# Age boxplot
plt.figure(figsize=(8, 3))
plt.boxplot(df_clean["age"], vert=False)
plt.title("Age Boxplot")
plt.xlabel("Age")
plt.show()

# Fare histogram
plt.figure(figsize=(8, 5))
plt.hist(df_clean["fare"], bins=30, edgecolor="black")
plt.title("Fare Distribution")
plt.xlabel("Fare")
plt.ylabel("Frequency")
plt.show()

# Fare boxplot
plt.figure(figsize=(8, 3))
plt.boxplot(df_clean["fare"], vert=False)
plt.title("Fare Boxplot")
plt.xlabel("Fare")
plt.show()

# IQR outlier function
def iqr_outlier_count(series):
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    count = ((series < lower) | (series > upper)).sum()
    return Q1, Q3, IQR, lower, upper, count

age_q1, age_q3, age_iqr, age_lower, age_upper, age_outliers = \
    iqr_outlier_count(df_clean["age"])

fare_q1, fare_q3, fare_iqr, fare_lower, fare_upper, fare_outliers = \
    iqr_outlier_count(df_clean["fare"])

# Fare statistics
fare_mean = df_clean["fare"].mean()
fare_median = df_clean["fare"].median()
fare_mode = df_clean["fare"].mode().iloc[0]

print("IQR OUTLIER ANALYSIS")
print("--------------------")
print(f"Age outliers: {age_outliers}")
print(f"Fare outliers: {fare_outliers}")

print("\nFare Statistics")
print("--------------------")
print(f"Mean: {fare_mean:.4f}")
print(f"Median: {fare_median:.4f}")
print(f"Mode: {fare_mode:.4f}")

# Skewness conclusion
if fare_mean > fare_median > fare_mode:
    conclusion = "Fare is right-skewed (mean > median > mode)."
elif fare_mean < fare_median < fare_mode:
    conclusion = "Fare is left-skewed (mean < median < mode)."
else:
    conclusion = "Fare distribution is approximately symmetric based on mean, median, and mode."

print("\nSkewness Conclusion")
print("--------------------")
print(conclusion)

In [ ]:
# ============================================
# TASK 4 — BIVARIATE ANALYSIS
# ============================================

# 1. Survival rate by SEX
survival_by_sex = df.groupby("sex")["survived"].mean() * 100

print("SURVIVAL RATE BY SEX")
print("--------------------")
print(survival_by_sex.round(2))
print()


# 2. Survival rate by PCLASS
survival_by_pclass = df.groupby("pclass")["survived"].mean() * 100

print("SURVIVAL RATE BY PCLASS")
print("-----------------------")
print(survival_by_pclass.round(2))
print()


# 3. Survival rate by SEX AND PCLASS
survival_by_sex_pclass = (
    df.groupby(["sex", "pclass"])["survived"]
    .mean() * 100
)

print("SURVIVAL RATE BY SEX AND PCLASS")
print("-------------------------------")
print(survival_by_sex_pclass.round(2))
print()


# 4. Correlation matrix — EXACTLY these six columns
corr_cols = [
    "survived",
    "pclass",
    "age",
    "sibsp",
    "parch",
    "fare"
]

corr_matrix = df[corr_cols].corr()

print("CORRELATION MATRIX")
print("------------------")
print(corr_matrix.round(3))
print()


# 5. Heatmap
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8, 6))

sns.heatmap(
    corr_matrix,
    annot=True,
    cmap="coolwarm",
    fmt=".2f",
    square=True
)

plt.title("Correlation Matrix Heatmap")
plt.tight_layout()
plt.show()


# 6.Find the two strongest UNIQUE correlations

import numpy as np

upper_triangle = corr_matrix.where(
    np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
)

corr_pairs = upper_triangle.stack()

top_two = corr_pairs.abs().sort_values(ascending=False).head(2)

print("TWO STRONGEST CORRELATIONS")
print("--------------------------")

for pair in top_two.index:
    value = corr_matrix.loc[pair[0], pair[1]]
    print(f"{pair[0]} ↔ {pair[1]} : {value:.3f}")

In [ ]:
# ============================================
# TASK 5 — MULTIVARIATE DATA STORY
# ============================================

import matplotlib.pyplot as plt
import seaborn as sns

# --------------------------------------------
# Chart 1: Survival by Sex and Passenger Class
# --------------------------------------------

plt.figure(figsize=(8, 5))

sns.barplot(
    data=df,
    x="sex",
    y="survived",
    hue="pclass",
    errorbar=None
)

plt.title("Survival Rate by Sex and Passenger Class")
plt.xlabel("Sex")
plt.ylabel("Survival Rate")
plt.ylim(0, 1)

plt.tight_layout()
plt.show()


# --------------------------------------------
# Chart 2: Age Distribution by Sex and Survival
# --------------------------------------------

plt.figure(figsize=(9, 5))

sns.boxplot(
    data=df,
    x="sex",
    y="age",
    hue="survived"
)

plt.title("Age Distribution by Sex and Survival")
plt.xlabel("Sex")
plt.ylabel("Age")

plt.tight_layout()
plt.show()


# --------------------------------------------
# Chart 3: Age vs Fare by Survival and Sex
# --------------------------------------------

plt.figure(figsize=(9, 6))

sns.scatterplot(
    data=df,
    x="age",
    y="fare",
    hue="survived",
    style="sex",
    alpha=0.7
)

plt.title("Age vs Fare by Survival and Sex")
plt.xlabel("Age")
plt.ylabel("Fare")

plt.tight_layout()
plt.show()


# --------------------------------------------
# Chart 4: Average Fare by Class and Sex
# --------------------------------------------

plt.figure(figsize=(8, 5))

sns.barplot(
    data=df,
    x="pclass",
    y="fare",
    hue="sex",
    errorbar=None
)

plt.title("Average Fare by Passenger Class and Sex")
plt.xlabel("Passenger Class")
plt.ylabel("Average Fare")

plt.tight_layout()
plt.show()

## Chart 1 — Survival Rate by Sex and Passenger Class

Female passengers generally had higher survival rates than male passengers across all passenger classes. Survival also varied across passenger classes, showing that both sex and passenger class are associated with survival.

## Chart 2 — Age Distribution by Sex and Survival

The age distributions are different between survivors and non-survivors for both males and females. The boxplot shows the spread and median age of each group and helps understand the relationship between age and survival.

## Chart 3 — Age vs Fare by Survival and Sex

The scatter plot shows the relationship between passenger age and fare for different survival statuses and sexes. Most passengers paid lower fares, while a few passengers paid very high fares.

## Chart 4 — Average Fare by Passenger Class and Sex

Average fare is much higher for first-class passengers than for second- and third-class passengers. The chart also shows differences in average fare between males and females within each passenger class.

In [ ]:
# 6. Standardize age and fare using Z-score

from sklearn.preprocessing import StandardScaler

# Create a copy of the cleaned dataset
df_scaled = df.copy()

# Before standardization
print("BEFORE STANDARDIZATION")
print("----------------------")
print("Age - Mean:", df_scaled["age"].mean())
print("Age - Std:", df_scaled["age"].std())
print("Fare - Mean:", df_scaled["fare"].mean())
print("Fare - Std:", df_scaled["fare"].std())

# Standardization
scaler = StandardScaler()
df_scaled[["age", "fare"]] = scaler.fit_transform(
    df_scaled[["age", "fare"]]
)

# After standardization
print("\nAFTER STANDARDIZATION")
print("---------------------")
print("Age - Mean:", df_scaled["age"].mean())
print("Age - Std:", df_scaled["age"].std())
print("Fare - Mean:", df_scaled["fare"].mean())
print("Fare - Std:", df_scaled["fare"].std())

## Task 6 — Standardization Check

The `age` and `fare` columns were standardized using Z-score standardization. After transformation, both columns have approximately mean 0 and standard deviation 1. This confirms that the standardization was successfully applied.

Note: This standardization is only an exploratory EDA check and is not used as the preprocessing step for the later modeling pipeline.

In [ ]:
# 7. Train/Test Split with Stratification

from sklearn.model_selection import train_test_split

# Separate features and target
X = df.drop("survived", axis=1)
y = df["survived"]

# Train-test split with stratification
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("TRAIN/TEST SPLIT")
print("----------------")
print("Training set shape:", X_train.shape)
print("Testing set shape:", X_test.shape)

print("\nTarget distribution:")
print("Overall:")
print(y.value_counts(normalize=True))

print("\nTraining:")
print(y_train.value_counts(normalize=True))

print("\nTesting:")
print(y_test.value_counts(normalize=True))

## Task 7 — Train/Test Split with Stratification

The cleaned dataset was divided into 80% training data and 20% testing data using a fixed random state. Stratification was applied to preserve the proportion of the target variable `survived` in both sets. The survival proportions in the training and testing sets are very close to the overall distribution, confirming that the split was performed correctly.

In [ ]:
# 8. Preprocessing Pipeline
# Fit preprocessing only on training data

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Columns to use for modeling
numeric_features = ["pclass", "age", "sibsp", "parch", "fare"]
categorical_features = ["sex", "embarked"]

# Numeric preprocessing:
# Missing values -> median
# Scaling -> StandardScaler
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Categorical preprocessing:
# Missing values -> most frequent
# Encoding -> One-Hot Encoding
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

# Combine preprocessing
preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

# FIT ONLY ON TRAINING DATA
X_train_processed = preprocessor.fit_transform(X_train)

# TRANSFORM TEST DATA ONLY
X_test_processed = preprocessor.transform(X_test)

print("PREPROCESSING COMPLETE")
print("----------------------")
print("Original training shape:", X_train.shape)
print("Original testing shape:", X_test.shape)
print("Processed training shape:", X_train_processed.shape)
print("Processed testing shape:", X_test_processed.shape)

print("\nPreprocessing was FIT on training data only.")
print("Test data was TRANSFORMED only.")

## Task 8 — Preprocessing Pipeline

Missing numeric values are handled using median imputation, while missing categorical values are replaced with the most frequent value. The categorical columns `sex` and `embarked` are one-hot encoded, and numeric features are standardized using `StandardScaler`. All preprocessing steps are fitted only on the training data, while the test data is transformed using the already-fitted preprocessing object to prevent data leakage.

In [ ]:
# Task 9 — Train 3 Classification Models

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# 1. Logistic Regression
logistic_model = LogisticRegression(max_iter=1000, random_state=42)
logistic_model.fit(X_train_processed, y_train)

# 2. Decision Tree
tree_model = DecisionTreeClassifier(random_state=42)
tree_model.fit(X_train_processed, y_train)

# 3. Random Forest
forest_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)
forest_model.fit(X_train_processed, y_train)

# Predictions
y_pred_logistic = logistic_model.predict(X_test_processed)
y_pred_tree = tree_model.predict(X_test_processed)
y_pred_forest = forest_model.predict(X_test_processed)

# Accuracy
print("MODEL ACCURACY")
print("----------------")
print("Logistic Regression:", accuracy_score(y_test, y_pred_logistic))
print("Decision Tree:", accuracy_score(y_test, y_pred_tree))
print("Random Forest:", accuracy_score(y_test, y_pred_forest))

# Classification reports
print("\nLOGISTIC REGRESSION")
print(classification_report(y_test, y_pred_logistic))

print("\nDECISION TREE")
print(classification_report(y_test, y_pred_tree))

print("\nRANDOM FOREST")
print(classification_report(y_test, y_pred_forest))

## Task 9 — Classification Models

Three classification models were trained using the preprocessed training data:

1. Logistic Regression
2. Decision Tree Classifier
3. Random Forest Classifier

The models were evaluated on the testing dataset using accuracy, precision, recall, and F1-score.

In [ ]:
# Task 10 — Model Evaluation

from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    roc_curve
)
import pandas as pd
import matplotlib.pyplot as plt

# Store models and predictions
models = {
    "Logistic Regression": logistic_model,
    "Decision Tree": tree_model,
    "Random Forest": forest_model
}

predictions = {
    "Logistic Regression": y_pred_logistic,
    "Decision Tree": y_pred_tree,
    "Random Forest": y_pred_forest
}

results = []

print("CONFUSION MATRICES")
print("===================")

for name in models:
    y_pred = predictions[name]
    model = models[name]

    cm = confusion_matrix(y_test, y_pred)

    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)

    y_prob = model.predict_proba(X_test_processed)[:, 1]
    auc = roc_auc_score(y_test, y_prob)

    results.append({
        "Model": name,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1 Score": f1,
        "ROC AUC": auc
    })

    print(f"\n{name}")
    print(cm)

# Comparison table
comparison_df = pd.DataFrame(results)

print("\nMODEL COMPARISON")
print("================")
print(comparison_df.round(4).to_string(index=False))

# ROC Curves
plt.figure(figsize=(8, 6))

for name in models:
    model = models[name]
    y_prob = model.predict_proba(X_test_processed)[:, 1]

    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc = roc_auc_score(y_test, y_prob)

    plt.plot(fpr, tpr, label=f"{name} (AUC = {auc:.3f})")

plt.plot([0, 1], [0, 1], linestyle="--")

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve — Model Comparison")
plt.legend()
plt.grid(True)
plt.show()

## Task 10 — Model Evaluation

All three classification models were evaluated using confusion matrix, accuracy, precision, recall, F1-score, and ROC-AUC.

A ROC curve was also plotted to compare the classification performance of Logistic Regression, Decision Tree, and Random Forest models.

In [ ]:
# TASK 11 — IMBALANCE HANDLING COMPARISON

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score, recall_score, f1_score
from imblearn.over_sampling import SMOTE

# -------------------------------
# 1. BASELINE MODEL
# -------------------------------
baseline_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

baseline_model.fit(X_train_processed, y_train)
y_pred_base = baseline_model.predict(X_test_processed)

base_precision = precision_score(y_test, y_pred_base)
base_recall = recall_score(y_test, y_pred_base)
base_f1 = f1_score(y_test, y_pred_base)


# -------------------------------
# 2. CLASS WEIGHT = BALANCED
# -------------------------------
balanced_model = RandomForestClassifier(
    n_estimators=100,
    class_weight="balanced",
    random_state=42
)

balanced_model.fit(X_train_processed, y_train)
y_pred_balanced = balanced_model.predict(X_test_processed)

balanced_precision = precision_score(y_test, y_pred_balanced)
balanced_recall = recall_score(y_test, y_pred_balanced)
balanced_f1 = f1_score(y_test, y_pred_balanced)


# -------------------------------
# 3. SMOTE
# -------------------------------
smote = SMOTE(random_state=42)

X_train_smote, y_train_smote = smote.fit_resample(
    X_train_processed,
    y_train
)

smote_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

smote_model.fit(X_train_smote, y_train_smote)
y_pred_smote = smote_model.predict(X_test_processed)

smote_precision = precision_score(y_test, y_pred_smote)
smote_recall = recall_score(y_test, y_pred_smote)
smote_f1 = f1_score(y_test, y_pred_smote)


# -------------------------------
# COMPARISON
# -------------------------------
print("IMBALANCE HANDLING COMPARISON")
print("=" * 70)

print(f"{'Method':<25}{'Precision':<15}{'Recall':<15}{'F1-Score':<15}")
print("-" * 70)

print(f"{'Baseline':<25}{base_precision:<15.4f}{base_recall:<15.4f}{base_f1:<15.4f}")

print(f"{'Class Weight Balanced':<25}"
      f"{balanced_precision:<15.4f}"
      f"{balanced_recall:<15.4f}"
      f"{balanced_f1:<15.4f}")

print(f"{'SMOTE':<25}"
      f"{smote_precision:<15.4f}"
      f"{smote_recall:<15.4f}"
      f"{smote_f1:<15.4f}")


# -------------------------------
# SMOTE CLASS DISTRIBUTION
# -------------------------------
print("\nCLASS DISTRIBUTION AFTER SMOTE")
print("-" * 40)
print(y_train_smote.value_counts())

### Task 11 — Imbalance Handling Comparison

The baseline Random Forest and class-weight-balanced Random Forest produced the same precision, recall, and F1-score.

SMOTE increased recall from 0.6957 to 0.7246, but precision decreased to 0.7463 and the F1-score to 0.7353.

SMOTE was applied only to the training data, while the original test data was kept unchanged to avoid data leakage.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV

# Random Forest with OOB enabled
rf = RandomForestClassifier(
    random_state=42,
    oob_score=True
)

# Hyperparameters to tune
param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [None, 5, 10],
    "max_features": ["sqrt", "log2"]
}

# Grid Search
grid_search = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    cv=5,
    scoring="f1",
    n_jobs=-1
)

grid_search.fit(X_train_processed, y_train)

# Best parameters
best_params = grid_search.best_params_

# Build final Random Forest with OOB enabled
best_rf = RandomForestClassifier(
    **best_params,
    random_state=42,
    oob_score=True
)

best_rf.fit(X_train_processed, y_train)

print("HYPERPARAMETER TUNING")
print("=====================")
print("Best Parameters:", best_params)
print("Best CV F1 Score:", grid_search.best_score_)
print("OOB Score:", best_rf.oob_score_)

### Task 12 — Hyperparameter Tuning

GridSearchCV was used to tune the Random Forest classifier using `n_estimators`, `max_depth`, and `max_features`.

The best parameter combination was selected using cross-validation based on F1-score. A final Random Forest model was then trained using the best parameters with OOB scoring enabled.

The OOB score provides an additional estimate of model performance using samples that were not selected during individual bootstrap training.

In [ ]:
# Task 13
# -------------------------------------------------------
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# Use the same dataset features to predict fare
X_reg = df.drop(columns=["fare"])
y_reg = df["fare"]

# Keep only numeric columns
X_reg = X_reg.select_dtypes(include=["number"])

# Handle missing values
X_reg = X_reg.fillna(X_reg.median())

# Train/test split
X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_reg,
    y_reg,
    test_size=0.2,
    random_state=42
)

# Linear Regression
reg_model = LinearRegression()
reg_model.fit(X_train_reg, y_train_reg)

# Predictions
y_pred_reg = reg_model.predict(X_test_reg)

# Metrics
mae = mean_absolute_error(y_test_reg, y_pred_reg)
rmse = np.sqrt(mean_squared_error(y_test_reg, y_pred_reg))
r2 = r2_score(y_test_reg, y_pred_reg)

# Adjusted R2
n = X_test_reg.shape[0]
p = X_test_reg.shape[1]

adjusted_r2 = 1 - (1 - r2) * (n - 1) / (n - p - 1)

print("REGRESSION SIDE-TASK")
print("===================")
print("MAE:", mae)
print("RMSE:", rmse)
print("R²:", r2)
print("Adjusted R²:", adjusted_r2)

In [ ]:
import matplotlib.pyplot as plt

# Calculate residuals
residuals = y_test_reg - y_pred_reg

# Residual plot
plt.figure(figsize=(8, 5))
plt.scatter(y_pred_reg, residuals, alpha=0.6)
plt.axhline(y=0, linestyle="--")

plt.xlabel("Predicted Fare")
plt.ylabel("Residuals")
plt.title("Residual Plot — Fare Regression")

plt.tight_layout()
plt.show()

### Task 13 — Regression Side-Task

A multiple linear regression model was used to predict passenger fare.

The model achieved an MAE of 19.83, RMSE of 30.79, R² of 0.389, and Adjusted R² of 0.371.

The residual plot was used to check whether the residuals were randomly distributed around zero. The residuals show some variation and spread across the predicted values, indicating that the model does not explain all fare variation and may contain some heteroscedasticity.

### Task 14 — Final Model Comparison

The classification models are compared using Accuracy, Precision, Recall, F1-score, and ROC-AUC. The regression model is evaluated separately using MAE, RMSE, R², and Adjusted R².

Classification and regression metrics are kept as separate metric groups because they measure different types of prediction performance.

Based on the classification metrics, the three classifiers show similar overall performance, with differences in precision, recall, F1-score, and ROC-AUC. The final model selection should consider the required balance between these metrics and the objective of the application.

### Final Model Comparison Table

| Model | Accuracy | Precision | Recall | F1-Score | ROC-AUC | MAE | RMSE | R² | Adjusted R² |
|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|
| Logistic Regression | 0.8045 | 0.7931 | 0.6667 | 0.7244 | 0.8437 | — | — | — | — |
| Decision Tree | 0.8156 | 0.7903 | 0.7101 | 0.7481 | 0.7904 | — | — | — | — |
| Random Forest | 0.8156 | 0.8000 | 0.6957 | 0.7442 | 0.8287 | — | — | — | — |
| Multiple Linear Regression | — | — | — | — | — | 19.83 | 30.79 | 0.389 | 0.371 |

### Final Recommendation

The classification models show similar accuracy, while their precision, recall, F1-score, and ROC-AUC values differ. Logistic Regression has the highest ROC-AUC, while Decision Tree has a slightly higher F1-score than the other classifiers.

For the regression task, the model achieved an R² of approximately 0.389, indicating that the selected features explain part of the variation in fare.

The final classifier should be selected according to the application's priority, such as ROC-AUC, recall, or F1-score.

In [ ]:
# Task 15
#--------------------------------------------------------
import joblib
from sklearn.pipeline import Pipeline

# Create complete pipeline
full_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", best_rf)
])

# Fit complete pipeline on raw training data
full_pipeline.fit(X_train, y_train)

# Save complete pipeline
joblib.dump(full_pipeline, "titanic_full_pipeline.joblib")

print("PIPELINE SAVED SUCCESSFULLY")
print("===========================")
print("File: titanic_full_pipeline.joblib")

# Reload pipeline
loaded_pipeline = joblib.load("titanic_full_pipeline.joblib")

# Test prediction on raw test data
sample_predictions = loaded_pipeline.predict(X_test.head(5))

print("\nRELOADED PIPELINE TEST")
print("======================")
print("Predictions:", sample_predictions)
print("Pipeline reload successful.")

Task 15 — Model Pipeline Saving and Reloading

The complete preprocessing pipeline and final Random Forest model were combined into a single pipeline and saved using joblib. The saved pipeline was successfully reloaded and tested on raw test data.

The reloaded pipeline produced predictions successfully, confirming that the saved artifact can be reused for prediction on new raw data without manually repeating the preprocessing steps.